# 04 - 문자열과 연산

이 노트북에서는:
- Kamailio cfg의 문자열 연산을 배웁니다
- 정수와 문자열의 자동 타입 변환을 이해합니다
- `$var`와 `$avp`의 차이를 심화 학습합니다
- 문자열 포매팅과 변수 치환을 연습합니다

---

## Kamailio의 데이터 타입

Kamailio cfg는 **동적 타이핑**을 사용합니다. 변수에 값을 할당할 때 타입이 자동으로 결정됩니다:
- `"hello"` → 문자열(string)
- `42` → 정수(integer)
- 타입은 할당 시점에 결정되며, 나중에 바뀔 수 있습니다

## 1. 문자열 기본 — 할당과 출력

In [ ]:
$var(greeting) = "Hello";
$var(target) = "Kamailio";
xlog("$var(greeting), $var(target)!");

## 2. 문자열 연결 (+)

`+` 연산자로 문자열을 연결할 수 있습니다.

In [ ]:
$var(user) = "1001";
$var(domain) = "example.com";
$var(uri) = "sip:" + $var(user) + "@" + $var(domain);
xlog("Constructed URI: $var(uri)");

## 3. 정수 연산

`+`는 정수 덧셈에도 사용됩니다. 피연산자가 모두 정수면 덧셈, 하나라도 문자열이면 문자열 연결입니다.

In [ ]:
$var(a) = 10;
$var(b) = 20;
$var(sum) = $var(a) + $var(b);
xlog("Sum: $var(a) + $var(b) = $var(sum)");

## 4. xlog로 변수 치환하기

`xlog()`는 `"..."` 안의 `$변수`를 자동으로 치환합니다. 이것이 Kamailio 디버깅의 기본입니다.

In [ ]:
%%sip INVITE
From: <sip:alice@example.com>;tag=abc123
To: <sip:bob@example.com>

In [ ]:
$var(caller) = $(fu{uri.user});
$var(callee) = $(tu{uri.user});
xlog("Caller: $var(caller), Callee: $var(callee)");
xlog("Request URI: $ru");
xlog("Call-ID: $ci");

## 5. $var vs $avp — 언제 뭘 쓰나?

| 구분 | `$var` | `$avp` |
|------|--------|--------|
| 범위 | 프로세스 내 (비공유) | 트랜잭션 (공유) |
| 저장 | 단일 값 | 스택 (여러 값) |
| 성능 | 빠름 | 상대적으로 느림 |
| 용도 | 임시 계산, 플래그 | 세션 데이터, 코더 정보 |

실무 팁: 단순 계산이나 임시 값은 `$var`, 트랜잭션 간 공유가 필요하면 `$avp`를 씁니다.

In [ ]:
# $var: 간단한 카운터
$var(retry_count) = 0;
$var(retry_count) = $var(retry_count) + 1;
xlog("Retry count: $var(retry_count)");

In [ ]:
# $avp: 트랜잭션 데이터
$avp(caller_domain) = $(fd{uri.host});
xlog("Caller domain stored in AVP: $avp(caller_domain)");

## 6. 변환 함수로 문자열 조작

변환 함수 `{s.upper}`, `{s.lower}`, `{s.len}` 등으로 문자열을 조작할 수 있습니다.

In [ ]:
$var(name) = "alice";
xlog("Original: $var(name)");
xlog("Upper: $(var(name){s.upper})");
xlog("Lower: $(var(name){s.lower})");
xlog("Length: $(var(name){s.len})");

## 7. URI에서 부분 추출하기

SIP URI에서 user, host, port를 분리하는 건 실무에서 매우 자주 씁니다.

In [ ]:
$var(raw_uri) = "sip:1001@10.0.0.1:5060";
xlog("Full URI: $var(raw_uri)");
xlog("User: $(var(raw_uri){uri.user})");
xlog("Host: $(var(raw_uri){uri.host})");
xlog("Port: $(var(raw_uri){uri.port})");

## 8. 비교 연산으로 라우팅 결정

문자열/정수 비교로 라우팅 분기를 결정합니다.

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=x1
To: <sip:2001@example.com>

In [ ]:
$var(dest) = $(ru{uri.user});

if ($var(dest) == "2001") {
    xlog("VIP customer — priority routing");
} else {
    xlog("Normal customer — standard routing");
}

---

### 요약

| 개념 | 문법 |
|------|------|
| 문자열 할당 | `$var(x) = "hello";` |
| 정수 할당 | `$var(n) = 42;` |
| 문자열 연결 | `"a" + "b"` 또는 `$var(x) + $var(y)` |
| 정수 덧셈 | `$var(a) + $var(b)` (둘 다 정수일 때) |
| 변수 치환 | `xlog("value: $var(x)");` |
| 대소문자 | `$(var(x){s.upper})`, `$(var(x){s.lower})` |
| 길이 | `$(var(x){s.len})` |
| URI 분해 | `{uri.user}`, `{uri.host}`, `{uri.port}` |
| 동등 비교 | `==`, `!=` |

다음 노트북: **Intermediate/01-transformations.ipynb** →